In [35]:
from dotenv import load_dotenv
load_dotenv()

True

In [36]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.summarize import load_summarize_chain
from langchain_classic.text_splitter import CharacterTextSplitter
from langchain_classic.docstore.document import Document

In [37]:
llm = ChatOpenAI(model="gpt-4o-mini",
    temperature=0.9)

In [38]:
with open('sample.txt') as f:
    data = f.read()

In [39]:
data

"India, officially known as the Republic of India, is a diverse and vibrant country located in South Asia. With a rich history spanning thousands of years, India is known for its cultural heritage, religious diversity, and vast landscapes. From the majestic Himalayas in the north to the serene backwaters of Kerala in the south, India encompasses a wide range of geographical features, including deserts, plains, mountains, and coastlines, making it a land of incredible natural beauty.\n\nIndia is the seventh-largest country by land area and the second-most populous country in the world, with a population exceeding 1.3 billion people. It is a federal parliamentary democratic republic, with a president as the head of state and a prime minister as the head of government. The country follows a multi-tiered administrative structure, with 28 states and 9 union territories, each having its own elected government.\n\nIndia has a rich cultural heritage that has evolved over thousands of years. It

In [40]:
text_split = CharacterTextSplitter()
texts = text_split.split_text(data)

In [41]:
texts

["India, officially known as the Republic of India, is a diverse and vibrant country located in South Asia. With a rich history spanning thousands of years, India is known for its cultural heritage, religious diversity, and vast landscapes. From the majestic Himalayas in the north to the serene backwaters of Kerala in the south, India encompasses a wide range of geographical features, including deserts, plains, mountains, and coastlines, making it a land of incredible natural beauty.\n\nIndia is the seventh-largest country by land area and the second-most populous country in the world, with a population exceeding 1.3 billion people. It is a federal parliamentary democratic republic, with a president as the head of state and a prime minister as the head of government. The country follows a multi-tiered administrative structure, with 28 states and 9 union territories, each having its own elected government.\n\nIndia has a rich cultural heritage that has evolved over thousands of years. I

In [42]:
docs = [Document(page_content = t) for t in texts]
docs

[Document(metadata={}, page_content="India, officially known as the Republic of India, is a diverse and vibrant country located in South Asia. With a rich history spanning thousands of years, India is known for its cultural heritage, religious diversity, and vast landscapes. From the majestic Himalayas in the north to the serene backwaters of Kerala in the south, India encompasses a wide range of geographical features, including deserts, plains, mountains, and coastlines, making it a land of incredible natural beauty.\n\nIndia is the seventh-largest country by land area and the second-most populous country in the world, with a population exceeding 1.3 billion people. It is a federal parliamentary democratic republic, with a president as the head of state and a prime minister as the head of government. The country follows a multi-tiered administrative structure, with 28 states and 9 union territories, each having its own elected government.\n\nIndia has a rich cultural heritage that has

In [43]:
chain = load_summarize_chain(llm, chain_type='map_reduce',verbose=True)
chain.run(docs)



> Entering new MapReduceDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Write a concise summary of the following:


"India, officially known as the Republic of India, is a diverse and vibrant country located in South Asia. With a rich history spanning thousands of years, India is known for its cultural heritage, religious diversity, and vast landscapes. From the majestic Himalayas in the north to the serene backwaters of Kerala in the south, India encompasses a wide range of geographical features, including deserts, plains, mountains, and coastlines, making it a land of incredible natural beauty.

India is the seventh-largest country by land area and the second-most populous country in the world, with a population exceeding 1.3 billion people. It is a federal parliamentary democratic republic, with a president as the head of state and a prime minister as the head of government. The country follows a multi-tiered administrative structure, with 28 st


> Finished chain.


> Entering new LLMChain chain...
Prompt after formatting:
Write a concise summary of the following:


"India, officially the Republic of India, is a diverse and culturally rich nation in South Asia, known for its vast landscapes ranging from the Himalayas to the backwaters of Kerala. It is the seventh-largest country and the second-most populous, with over 1.3 billion people. A federal parliamentary democracy, India has 28 states and 9 union territories. Its cultural heritage includes multiple religions—such as Hinduism, Islam, and Christianity—and vibrant festivals like Diwali and Holi. Historically, India has seen the rise and fall of ancient civilizations and empires, culminating in independence from British rule on August 15, 1947. Economically, it has shifted to a service-oriented and industrialized economy, excelling in sectors like IT, pharmaceuticals, and textiles while facing challenges like poverty and inequality. Education and the film industry, particul

'India, officially the Republic of India, is a culturally rich and diverse nation in South Asia, known for its vast landscapes and significant population of over 1.3 billion. It operates as a federal parliamentary democracy comprising 28 states and 9 union territories. Home to multiple religions and vibrant festivals, India gained independence from British rule in 1947. The economy has transitioned to being service-oriented and industrialized, excelling in IT, pharmaceuticals, and textiles, though challenges like poverty remain. Indian cuisine and Bollywood contribute to its global presence, while tourism flourishes through iconic sites like the Taj Mahal. The Indian rupee (₹) is the official currency, and the nation is recognized for its UNESCO World Heritage Sites and advancements in space through ISRO. Overall, India is a dynamic blend of ancient traditions and modern advancements.'

HTTP Request

In [44]:
from langchain_classic.chains import LLMRequestsChain, LLMChain

In [45]:
template = """
Question: {Query}
Webpage content: {requests_result}

Answer the question using the webpage content.
"""

prompt = PromptTemplate(
    input_variable = ["Query","request_result"],
    template= template
)


In [46]:
llm_chains = LLMChain(llm=llm, prompt = prompt)

In [47]:
req_chains = LLMRequestsChain(llm_chain = llm_chains)

In [54]:
ques = "What is the capital  of INdia?"
input = {
    "Query":ques,
    'url':"https://www.google.com/search?q="+ques.replace(' ','+')
}

In [55]:
print(req_chains.input_keys)

['url']


In [56]:
print(req_chains.input_keys)
print(req_chains.llm_chain.input_keys)

['url']
['Query', 'requests_result']


In [57]:
req_chains(input)

{'Query': 'What is the capital  of INdia?',
 'url': 'https://www.google.com/search?q=What+is+the+capital++of+INdia?',
 'output': 'The webpage content does not provide the information needed to answer the question about the capital of India. However, the capital of India is New Delhi.'}

In [58]:
import inspect

In [59]:
print(inspect.getsource(chain._call))

    def _call(
        self,
        inputs: dict[str, list[Document]],
        run_manager: CallbackManagerForChainRun | None = None,
    ) -> dict[str, str]:
        """Prepare inputs, call combine docs, prepare outputs."""
        _run_manager = run_manager or CallbackManagerForChainRun.get_noop_manager()
        docs = inputs[self.input_key]
        # Other keys are assumed to be needed for LLM prediction
        other_keys = {k: v for k, v in inputs.items() if k != self.input_key}
        output, extra_return_dict = self.combine_docs(
            docs,
            callbacks=_run_manager.get_child(),
            **other_keys,
        )
        extra_return_dict[self.output_key] = output
        return extra_return_dict

